## 1. Install Dependencies

All agents use Python stdlib only — no packages to install.

In [1]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    print("Colab: no additional packages required — stdlib only.")
else:
    print("Local: stdlib only — no installation needed.")

Local: stdlib only — no installation needed.


## 2. Repo Setup

Clone repo in Colab (`src/` files already in repo), add `src/` to sys.path.

In [1]:
import os, sys, warnings, json
from pathlib import Path

warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not Path("/content/NLP-Lab-works").exists():
        os.system("git clone https://github.com/DanylchenkoKateryna/NLP-Lab-works.git /content/NLP-Lab-works")
    ROOT = Path("/content/NLP-Lab-works")
else:
    p = Path.cwd()
    ROOT = None
    for _ in range(6):
        if (p / "src" / "crew_workflow.py").exists():
            ROOT = p
            break
        p = p.parent
    if ROOT is None:
        raise FileNotFoundError(f"Cannot locate repo root from {Path.cwd()}")

os.chdir(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

print(f"ROOT: .../{ROOT.name}")
print("Source modules ready.")

ROOT: .../lab1
Source modules ready.


## 3. Test Cases

In [1]:
from eval_crew import TEST_CASES, SINGLE_AGENT_BASELINE

print(f"Test cases: {len(TEST_CASES)}")
print()
print(f'{"#":<4} {"case_id":<12} {"scenario":<30} note')
print("-" * 90)
for i, tc in enumerate(TEST_CASES, 1):
    print(f'{i:<4} {tc["case_id"]:<12} {tc["scenario"]:<30} {tc["note"][:40]}')

Test cases: 10

#    case_id      scenario                       note
------------------------------------------------------------------------------------------
1    case_001     simple                         Golden path — clear electronics text wit
2    case_002     missing_required_field         No category keywords, no entities — revi
3    case_003     ambiguous_entity               Equal keyword counts force ambiguous rou
4    case_004     relative_date                  Relative dates cannot be normalized with
5    case_005     hallucination_prone            An LLM extractor would add Intel; rule-b
6    case_006     simulated_hallucination        Pre-extracted with HP hallucination — re
7    case_007     fallback_needed                Empty input — must produce structured sa
8    case_008     reviewer_rejects_then_accepts  Clear atheism text — reviewer verifies c
9    case_009     repair_helps                   Repair corrects wrong category — classic
10   case_010     repair_fail

## 4. Agent Role Definitions

Four agents in the crew:
- **TriagerAgent** `src/agents.py` — keyword analysis, route selection
- **ExtractorAgent** `src/agents.py` — structured extraction via tools.py
- **ReviewerAgent** `src/reviewer.py` — schema + consistency + hallucination checks
- **RepairAgent / FallbackHandler** `src/fallback.py` — fix issues or safe failure

In [1]:
from agents import TriagerAgent, ExtractorAgent, TriageResult
from reviewer import ReviewerAgent
from fallback import RepairAgent, FallbackHandler

# Demo Triager on various inputs
triager = TriagerAgent()
demos = [
    ("Intel microprocessor circuit voltage signal processing.", "clear electronics"),
    ("Jesus Christ faith Christian Bible.",                     "clear christian"),
    ("Richard Dawkins atheism evolution rational.",             "clear atheism"),
    ("Jesus faith transistor circuit electronics microprocessor.", "ambiguous tie"),
    ("",                                                        "empty input"),
]
print(f'{"input_type":<20} {"route":<22} {"difficulty":<10} notes')
print("-" * 80)
for text, label in demos:
    t = triager.triage(text, "demo")
    print(f'{label:<20} {t.route:<22} {t.difficulty:<10} {t.notes[:35]}')

input_type           route                  difficulty notes
--------------------------------------------------------------------------------
clear electronics    electronics_schema     easy       electronics=5; very short text
clear christian      religious_schema       easy       christian=5; very short text
clear atheism        atheism_schema         easy       atheism=5; very short text
ambiguous tie        electronics_schema     medium     electronics=4; christian=2; very sh
empty input          unknown                trivial    Empty or blank input — cannot route


In [1]:
# Demo Extractor
extractor = ExtractorAgent()
demo_triage = TriageResult(
    task_type="electronics_classification",
    route="electronics_schema",
    expected_fields=["category","persons","organizations","locations","dates"],
    difficulty="easy",
    notes="3 electronics keywords",
)
demo_text = "Intel released the 4004 microprocessor in November 1971. The circuit changed computing."
extraction = extractor.extract(demo_text, demo_triage, "demo")
print("Extractor output:")
for k, v in extraction.items():
    print(f"  {k}: {v}")

Extractor output:
  category: sci.electronics
  persons: []
  organizations: ['Intel']
  locations: []
  dates: ['November 1971']
  confidence_note: high confidence (100%); 2 entities extracted


In [1]:
# Demo Reviewer
reviewer = ReviewerAgent()

# Good extraction
review_good = reviewer.review(demo_text, demo_triage, extraction, "demo")
print(f"Reviewer (good): verdict={review_good.verdict}  issues={review_good.issues}")

# Hallucinated entity
bad_ext = dict(extraction)
bad_ext["organizations"] = ["Intel", "Hewlett-Packard"]
review_bad = reviewer.review(demo_text, demo_triage, bad_ext, "demo")
print(f"Reviewer (hallucination): verdict={review_bad.verdict}")
for issue in review_bad.issues:
    print(f"  issue: {issue}")

Reviewer (good): verdict=accept  issues=[]
Reviewer (hallucination): verdict=fallback_needed
  issue: {'field': 'organizations', 'problem': "hallucinated entity 'Hewlett-Packard' not found in text", 'entity': 'Hewlett-Packard'}


## 5. Delegation Rules

```
1. Triager is always called first
2. Extractor is always called after Triager
3. Reviewer always checks Extractor output

if verdict == 'accept':
    -> status = 'accepted'

elif verdict == 'repair_needed':
    -> RepairAgent (fix issues)
    -> Reviewer (re-check)
    -> if accept: 'accepted_after_repair'
    -> else:      'manual_review'

elif verdict == 'fallback_needed':
    -> FallbackHandler -> 'fallback'

elif verdict == 'manual_review':
    -> 'manual_review'  (no more agents)
```

In [1]:
# Show delegation in action for three cases
from crew_workflow import CrewWorkflow
from eval_crew import TEST_CASES

crew = CrewWorkflow()

for case_id in ["case_001", "case_006", "case_009"]:
    tc = next(t for t in TEST_CASES if t["case_id"] == case_id)
    r = crew.run(tc["input"], tc["case_id"], tc.get("pre_extracted"))
    print(f"[{case_id}] agents: {r.agents_called}")
    print(f"  status: {r.status}")
    print()

[case_001] agents: ['Triager', 'Extractor', 'Reviewer']
  status: accepted

[case_006] agents: ['Triager', 'Extractor', 'Reviewer', 'FallbackHandler']
  status: fallback

[case_009] agents: ['Triager', 'Extractor', 'Reviewer', 'RepairAgent', 'Reviewer (re-check)']
  status: accepted_after_repair


## 6. Single-agent Baseline (LR12)

In [1]:
from eval_crew import run_single_agent_baseline

baseline_results = run_single_agent_baseline(TEST_CASES)

print("Single-agent baseline:")
print("=" * 80)
for r in baseline_results:
    bl = SINGLE_AGENT_BASELINE.get(r["case_id"], {})
    issue = " *** ISSUE ***" if any(x in bl.get("note","") for x in ("WRONG","MISSED","HALLUCINATION","Partial")) else ""
    print(f'[{r["case_id"]}] cat={str(r["category"]):<28}{issue}')
    print(f'  note: {bl.get("note","")[:75]}')
    print()

Single-agent baseline:
[case_001] cat=sci.electronics             
  note: Correct — baseline handles simple case well

[case_002] cat=unknown                     
  note: Correct — no signal, returns empty

[case_003] cat=ambiguous                    *** ISSUE ***
  note: WRONG — baseline picks christian without detecting electronics tie

[case_004] cat=soc.religion.christian       *** ISSUE ***
  note: Partial — baseline does not flag relative dates at all

[case_005] cat=sci.electronics             
  note: Correct — minimal extraction, no hallucinations

[case_006] cat=sci.electronics              *** ISSUE ***
  note: HALLUCINATION MISSED — baseline accepts HP without checking text

[case_007] cat=unknown                     
  note: Correct — empty input returns error

[case_008] cat=alt.atheism                 
  note: Correct — baseline handles clear atheism text well

[case_009] cat=soc.religion.christian       *** ISSUE ***
  note: WRONG CATEGORY — baseline returns same wrong

## 7. Multi-agent Crew Workflow

In [1]:
# Smoke test
smoke = crew.run(
    "Intel released the microprocessor in November 1971. The circuit changed computing.",
    "smoke_test",
)
print("Smoke test:")
print(f"  status        : {smoke.status}")
print(f"  agents called : {smoke.agents_called}")
print(f"  final category: {smoke.final_output.get('category')}")
print(f"  organizations : {smoke.final_output.get('organizations')}")
print(f"  dates         : {smoke.final_output.get('dates')}")

Smoke test:
  status        : accepted
  agents called : ['Triager', 'Extractor', 'Reviewer']
  final category: sci.electronics
  organizations : ['Intel']
  dates         : ['November 1971']


## 8. Reviewer Consistency Checks

ReviewerAgent checks five conditions independently:
1. Schema validity — all required fields present
2. Category consistency — re-run keyword scorer, compare to extracted category
3. Hallucination — entity not found as substring in source text
4. Completeness — long text but empty entity lists
5. Relative dates — temporal expressions requiring normalization

In [1]:
from reviewer import ReviewerAgent
from agents import TriagerAgent

rev = ReviewerAgent()
tri = TriagerAgent()

checks = [
    ("Intel circuit microprocessor.",
     {"category":"sci.electronics","persons":[],"organizations":["Intel"],
      "locations":[],"dates":[],"confidence_note":"ok"},
     "consistent"),
    ("Intel circuit microprocessor.",
     {"category":"alt.atheism","persons":[],"organizations":["Intel"],
      "locations":[],"dates":[],"confidence_note":"wrong"},
     "inconsistent category"),
    ("Intel circuit microprocessor.",
     {"category":"sci.electronics","persons":[],"organizations":["Intel","NASA"],
      "locations":[],"dates":[],"confidence_note":"halluc"},
     "hallucinated NASA"),
    ("Pope visits Poland next month.",
     {"category":"soc.religion.christian","persons":["Pope John Paul II"],
      "organizations":[],"locations":["Poland"],"dates":[],"confidence_note":"ok"},
     "relative date"),
    ("", {"_extraction_failed": True}, "empty input"),
]

print(f'{"description":<22} {"verdict":<20} issues')
print("-" * 80)
for text, ext, desc in checks:
    triage = tri.triage(text, "demo")
    result = rev.review(text, triage, ext, "demo")
    issue_str = "; ".join(i["problem"][:38] for i in result.issues) or "none"
    print(f'{desc:<22} {result.verdict:<20} {issue_str}')

description            verdict              issues
--------------------------------------------------------------------------------
consistent             accept               none
inconsistent category  repair_needed        keyword evidence suggests 'sci.electro
hallucinated NASA      fallback_needed      hallucinated entity 'NASA' not found i
relative date          fallback_needed      hallucinated entity 'Pope John Paul II; relative date 'next month' found — nor
empty input            fallback_needed      extraction failed — empty input


## 9. Fallback Logic

Two fallback strategies:
- **RepairAgent**: targeted fix for known issues (wrong category, hallucinations, relative dates)
- **FallbackHandler**: full re-extraction or safe structured failure

In [1]:
from fallback import RepairAgent, FallbackHandler

rep = RepairAgent()
fb  = FallbackHandler()

# Repair: wrong category
text_r = "Pope John Paul II visited Poland in 1979. The Catholic Church celebrated."
wrong_ext = {"category":"alt.atheism","persons":["Pope John Paul II"],
             "organizations":["Catholic Church"],"locations":["Poland"],
             "dates":["1979"],"confidence_note":"wrong"}
triage_r  = tri.triage(text_r, "demo")
review_r  = rev.review(text_r, triage_r, wrong_ext, "demo")
repaired  = rep.repair(text_r, wrong_ext, review_r, "demo")
print("Repair (wrong category):")
print(f"  issue  : {review_r.issues[0]['problem']}")
print(f"  before : {wrong_ext['category']}")
print(f"  after  : {repaired['category']}")
print()

# Fallback: hallucination
text_f   = "Intel designed a new chip."
bad_ext  = {"category":"sci.electronics","persons":[],
            "organizations":["Intel","Hewlett-Packard"],
            "locations":[],"dates":[],"confidence_note":"halluc"}
triage_f = tri.triage(text_f, "demo")
review_f = rev.review(text_f, triage_f, bad_ext, "demo")
fb_result = fb.handle(text_f, bad_ext, review_f, "demo")
print("Fallback (hallucination):")
print(f"  reviewer : {review_f.verdict}")
print(f"  strategy : {fb_result.get('_fallback_strategy')}")
print(f"  orgs     : {fb_result.get('organizations')}  # HP removed")
print()

# Repair: relative date
text_d = "Pope John Paul II will visit Poland next month. Vatican announced upcoming trip."
ext_d  = {"category":"soc.religion.christian","persons":["Pope John Paul II"],
          "organizations":["Vatican"],"locations":["Poland"],
          "dates":[],"confidence_note":"ok"}
triage_d   = tri.triage(text_d, "demo")
review_d   = rev.review(text_d, triage_d, ext_d, "demo")
repaired_d = rep.repair(text_d, ext_d, review_d, "demo")
print("Repair (relative date):")
print(f"  issue              : {review_d.issues[0]['problem']}")
print(f"  has_relative_date  : {repaired_d.get('has_relative_date')}")
print(f"  needs_manual_review: {repaired_d.get('needs_manual_review')}")

Repair (wrong category):
  issue  : keyword evidence suggests 'soc.religion.christian' but extracted 'alt.atheism'
  before : alt.atheism
  after  : soc.religion.christian

Fallback (hallucination):
  reviewer : fallback_needed
  strategy : rule_based_reextraction
  orgs     : ['Intel']  # HP removed

Repair (relative date):
  issue              : relative date 'next month' found — normalization needed
  has_relative_date  : True
  needs_manual_review: True


## 10. Run 10 Test Cases

In [1]:
crew_results = []

for tc in TEST_CASES:
    r = crew.run(tc["input"], tc["case_id"], tc.get("pre_extracted"))
    crew_results.append(r)

print("Crew results:")
print("=" * 90)
for r in crew_results:
    print(r.summary_line())

Crew results:
[case_001] accepted                     | agents=3 | cat=sci.electronics             
[case_002] accepted                     | agents=3 | cat=unknown                     
[case_003] manual_review                | agents=5 | cat=ambiguous                    [FALLBACK]
[case_004] accepted_after_repair        | agents=5 | cat=soc.religion.christian       [FALLBACK]
[case_005] accepted                     | agents=3 | cat=sci.electronics             
[case_006] fallback                     | agents=4 | cat=sci.electronics              [FALLBACK]
[case_007] fallback                     | agents=4 | cat=?                            [FALLBACK]
[case_008] accepted                     | agents=3 | cat=alt.atheism                 
[case_009] accepted_after_repair        | agents=5 | cat=soc.religion.christian       [FALLBACK]
[case_010] manual_review                | agents=5 | cat=ambiguous                    [FALLBACK]


In [1]:
# Side-by-side: baseline vs crew
print(f'{"case_id":<12} {"scenario":<25} {"baseline_issue":<35} crew_status')
print("-" * 100)
for r, tc in zip(crew_results, TEST_CASES):
    bl_note = SINGLE_AGENT_BASELINE.get(r.case_id, {}).get("note","")[:33]
    print(f'{r.case_id:<12} {tc["scenario"]:<25} {bl_note:<35} {r.status}')

case_id      scenario                  baseline_issue                      crew_status
----------------------------------------------------------------------------------------------------
case_001     simple                    Correct — baseline handles simple   accepted
case_002     missing_required_field    Correct — no signal, returns empt   accepted
case_003     ambiguous_entity          WRONG — baseline picks christian    manual_review
case_004     relative_date             Partial — baseline does not flag    accepted_after_repair
case_005     hallucination_prone       Correct — minimal extraction, no    accepted
case_006     simulated_hallucination   HALLUCINATION MISSED — baseline a   fallback
case_007     fallback_needed           Correct — empty input returns err   fallback
case_008     reviewer_rejects_then_accepts Correct — baseline handles clear    accepted
case_009     repair_helps              WRONG CATEGORY — baseline returns   accepted_after_repair
case_010     repair_f

## 11. Crew Logs

Save `docs/crew_logs_lab13.jsonl` — one JSON line per test case.

In [1]:
log_path = ROOT / "docs" / "crew_logs_lab13.jsonl"

with open(log_path, "w", encoding="utf-8") as f:
    for r in crew_results:
        f.write(json.dumps(r.to_log_dict(), ensure_ascii=False) + "\n")

lines = log_path.read_text(encoding="utf-8").strip().splitlines()
print(f"Saved {len(lines)} log entries to docs/crew_logs_lab13.jsonl")

sample = json.loads(lines[0])
print()
print(f"Sample entry keys: {list(sample.keys())}")
print(f"case_id: {sample['case_id']}")
print(f"status:  {sample['status']}")
print(f"agents:  {sample['agents_called']}")

Saved 10 log entries to docs/crew_logs_lab13.jsonl

Sample entry keys: ['case_id', 'input', 'triager_output', 'extractor_output', 'reviewer_output', 'fallback_triggered', 'fallback_output', 'final_output', 'status', 'agents_called']
case_id: case_001
status:  accepted
agents:  ['Triager', 'Extractor', 'Reviewer']


## 12. Metrics

In [1]:
from eval_crew import compute_crew_metrics, compute_baseline_metrics

crew_m = compute_crew_metrics(crew_results)
base_m = compute_baseline_metrics(baseline_results, TEST_CASES)

sep = "-" * 50
print("=" * 50)
print("  Multi-agent Crew Metrics")
print("=" * 50)
print(f"  Total test cases          : {crew_m['total_cases']}")
print(sep)
print(f"  Valid final output rate   : {crew_m['valid_final_output_rate']:.1%}  ({crew_m['valid_final_count']}/{crew_m['total_cases']})")
print(f"  Reviewer catch rate       : {crew_m['reviewer_catch_rate']:.1%}  ({crew_m['reviewer_caught']}/{crew_m['real_problems']})")
print(f"  Fallback activation rate  : {crew_m['fallback_activation_rate']:.1%}  ({crew_m['fallback_triggered']}/{crew_m['total_cases']})")
print(f"  Fallback success rate     : {crew_m['fallback_success_rate']:.1%}  ({crew_m['fallback_success']}/{crew_m['fallback_triggered']})")
print(f"  Manual review rate        : {crew_m['manual_review_rate']:.1%}  ({crew_m['manual_review_count']}/{crew_m['total_cases']})")
print(f"  Avg agents per case       : {crew_m['avg_agents_per_case']}")
print(sep)
print(f"  Status distribution       : {crew_m['status_distribution']}")
print()
print("=" * 50)
print("  Single-agent Baseline")
print("=" * 50)
print(f"  Baseline accuracy         : {base_m['baseline_accuracy']:.1%}  ({base_m['baseline_correct']}/{base_m['total_cases']})")
print(f"  Wrong category cases      : {base_m['wrong_category_count']}")
print(f"  Hallucinations missed     : {base_m['hallucination_missed']}")
print()
print("=" * 50)
print("  Crew vs Baseline Summary")
print("=" * 50)
crew_ok = crew_m["valid_final_count"]
base_ok = base_m["baseline_correct"]
print(f"  Crew valid output  : {crew_ok}/10 = {crew_ok * 10:.0f}%")
print(f"  Baseline correct   : {base_ok}/10 = {base_ok * 10:.0f}%")
print(f"  Crew improvement   : +{crew_ok - base_ok} cases")

  Multi-agent Crew Metrics
  Total test cases          : 10
--------------------------------------------------
  Valid final output rate   : 90.0%  (9/10)
  Reviewer catch rate       : 100.0%  (6/6)
  Fallback activation rate  : 60.0%  (6/10)
  Fallback success rate     : 50.0%  (3/6)
  Manual review rate        : 20.0%  (2/10)
  Avg agents per case       : 4.0
--------------------------------------------------
  Status distribution       : {'accepted': 4, 'manual_review': 2, 'accepted_after_repair': 2, 'fallback': 2}

  Single-agent Baseline
  Baseline accuracy         : 60.0%  (6/10)
  Wrong category cases      : 3
  Hallucinations missed     : 1

  Crew vs Baseline Summary
  Crew valid output  : 9/10 = 90%
  Baseline correct   : 6/10 = 60%
  Crew improvement   : +3 cases


## 13. Error Analysis

Structured analysis of all 10 cases.

Error categories: `none`, `completeness_warning`, `unresolvable_ambiguity`,
`relative_date_detected`, `hallucinated_entity`, `empty_input`, `wrong_category`

In [1]:
ERROR_ANALYSIS = [
    {"case_id":"case_001","scenario":"simple","error_category":"none",
     "triager":"electronics_schema/easy","extractor":"sci.electronics+Intel+Nov 1971",
     "reviewer":"accept","fallback":"none","final_status":"accepted",
     "notes":"Golden path. All agents agree.","possible_fix":"N/A"},
    {"case_id":"case_002","scenario":"missing_required_field","error_category":"completeness_warning",
     "triager":"unknown_schema/hard","extractor":"unknown+empty entities",
     "reviewer":"accept","fallback":"none","final_status":"accepted",
     "notes":"No keyword signal. Graceful empty result.","possible_fix":"N/A"},
    {"case_id":"case_003","scenario":"ambiguous_entity","error_category":"unresolvable_ambiguity",
     "triager":"mixed_schema/hard(tie)","extractor":"ambiguous",
     "reviewer":"repair_needed","fallback":"repair tries, still tied, needs_manual",
     "final_status":"manual_review",
     "notes":"Tied christian=4/electronics=4. Repair cannot break tie.",
     "possible_fix":"LLM disambiguation"},
    {"case_id":"case_004","scenario":"relative_date","error_category":"relative_date_detected",
     "triager":"religious_schema/medium","extractor":"soc.religion.christian+entities OK",
     "reviewer":"repair_needed(relative date)","fallback":"repair marks needs_manual_review",
     "final_status":"accepted_after_repair",
     "notes":"Relative dates flagged, repair acknowledges, re-review accepts.",
     "possible_fix":"Inject current date"},
    {"case_id":"case_005","scenario":"hallucination_prone","error_category":"none",
     "triager":"electronics_schema/hard(short)","extractor":"sci.electronics+no entities",
     "reviewer":"accept","fallback":"none","final_status":"accepted",
     "notes":"Short text. Rule-based stays clean (LLM would add Intel).",
     "possible_fix":"N/A"},
    {"case_id":"case_006","scenario":"simulated_hallucination","error_category":"hallucinated_entity",
     "triager":"electronics_schema/easy","extractor":"[Intel, HP] -- HP not in text",
     "reviewer":"fallback_needed","fallback":"FallbackHandler reextracts, only Intel",
     "final_status":"fallback",
     "notes":"Reviewer catches HP hallucination. Fallback cleans it.",
     "possible_fix":"N/A"},
    {"case_id":"case_007","scenario":"fallback_needed","error_category":"empty_input",
     "triager":"unknown/trivial","extractor":"_extraction_failed=True",
     "reviewer":"fallback_needed","fallback":"safe_failure structured error",
     "final_status":"fallback",
     "notes":"Empty input. Structured safe failure output.",
     "possible_fix":"Pre-validate input"},
    {"case_id":"case_008","scenario":"reviewer_rejects_then_accepts","error_category":"none",
     "triager":"atheism_schema/medium","extractor":"alt.atheism+Dawkins+2006",
     "reviewer":"accept","fallback":"none","final_status":"accepted",
     "notes":"Reviewer confirms atheism keywords dominate. Correct.",
     "possible_fix":"N/A"},
    {"case_id":"case_009","scenario":"repair_helps","error_category":"wrong_category",
     "triager":"religious_schema/medium","extractor":"alt.atheism(WRONG)+Pope+entities OK",
     "reviewer":"repair_needed(inconsistency)","fallback":"repair corrects to soc.religion.christian",
     "final_status":"accepted_after_repair",
     "notes":"Repair corrects wrong category. Re-review accepts.",
     "possible_fix":"N/A"},
    {"case_id":"case_010","scenario":"repair_fails_manual","error_category":"unresolvable_ambiguity",
     "triager":"mixed_schema/hard(tie)","extractor":"ambiguous",
     "reviewer":"repair_needed","fallback":"repair tries, still tied, needs_manual",
     "final_status":"manual_review",
     "notes":"Ambiguity persists after repair. Manual review required.",
     "possible_fix":"LLM disambiguation"},
]

print("Error Analysis -- all 10 cases")
print("=" * 95)
print(f'{"#":<3} {"case_id":<12} {"error_category":<30} {"final_status":<25} notes')
print("-" * 95)
for i, e in enumerate(ERROR_ANALYSIS, 1):
    print(f'{i:<3} {e["case_id"]:<12} {e["error_category"]:<30} {e["final_status"]:<25} {e["notes"][:35]}')

Error Analysis -- all 10 cases
#   case_id      error_category                 final_status              notes
-----------------------------------------------------------------------------------------------
1   case_001     none                           accepted                  Golden path. All agents agree.
2   case_002     completeness_warning           accepted                  No keyword signal. Graceful empty r
3   case_003     unresolvable_ambiguity         manual_review             Tied christian=4/electronics=4. Can
4   case_004     relative_date_detected         accepted_after_repair     Relative dates flagged, repair ackn
5   case_005     none                           accepted                  Short text. Rule-based stays clean.
6   case_006     hallucinated_entity            fallback                  Reviewer catches HP hallucination. 
7   case_007     empty_input                    fallback                  Empty input. Structured safe failur
8   case_008     none       

In [1]:
from collections import Counter
cat_counts = Counter(e["error_category"] for e in ERROR_ANALYSIS)
print("Error category distribution:")
for cat, cnt in sorted(cat_counts.items(), key=lambda x: -x[1]):
    print(f"  {cat:<35}: {cnt}")
print()
print("Cases where crew outperformed baseline:")
improvements = ["case_003", "case_006", "case_009", "case_010"]
for r in crew_results:
    if r.case_id in improvements:
        bl = SINGLE_AGENT_BASELINE.get(r.case_id, {})
        print(f"  [{r.case_id}] baseline: {bl.get('note','')[:55]}")
        print(f"  [{r.case_id}] crew    : {r.status}")
        print()

Error category distribution:
  none                               : 3
  unresolvable_ambiguity             : 2
  relative_date_detected             : 1
  hallucinated_entity                : 1
  empty_input                        : 1
  wrong_category                     : 1

Cases where crew outperformed baseline:
  [case_003] baseline: WRONG — baseline picks christian without detecting elec
  [case_003] crew    : manual_review

  [case_006] baseline: HALLUCINATION MISSED — baseline accepts HP without chec
  [case_006] crew    : fallback

  [case_009] baseline: WRONG CATEGORY — baseline returns same wrong 'alt.athei
  [case_009] crew    : accepted_after_repair

  [case_010] baseline: WRONG — baseline picks christian, ignores electronics t
  [case_010] crew    : manual_review


## 14. Generate `docs/audit_summary_lab13.md`

In [1]:
import datetime

audit_lines = [
    "# Audit Summary - Lab 13: Multi-agent Crew", "",
    f"**Date:** {datetime.date.today()}", "",
    "## 1. Use Case",
    "Multi-agent crew for 20 Newsgroups post classification and entity extraction.",
    "Extends single-agent pipeline (LR12) with routing, review, repair, and fallback.", "",
    "## 2. Agents Implemented",
    "- TriagerAgent   - keyword-based routing",
    "- ExtractorAgent - extraction via tools.py",
    "- ReviewerAgent  - schema/consistency/hallucination/date checks",
    "- RepairAgent    - targeted fixes",
    "- FallbackHandler - re-extraction or safe failure", "",
    "## 3. Test Cases",
    "10 cases: simple, missing_required_field, ambiguous_entity, relative_date,",
    "hallucination_prone, simulated_hallucination, fallback_needed,",
    "reviewer_rejects_then_accepts, repair_helps, repair_fails_manual.", "",
    f"## 4. Valid Final Output Rate: {crew_m['valid_final_count']}/{crew_m['total_cases']} = {crew_m['valid_final_output_rate']:.1%}", "",
    f"## 5. Reviewer Catch Rate: {crew_m['reviewer_caught']}/{crew_m['real_problems']} = {crew_m['reviewer_catch_rate']:.1%}", "",
    f"## 6. Fallback Activation Rate: {crew_m['fallback_triggered']}/{crew_m['total_cases']} = {crew_m['fallback_activation_rate']:.1%}", "",
    f"## 7. Fallback Success Rate: {crew_m['fallback_success']}/{crew_m['fallback_triggered']} = {crew_m['fallback_success_rate']:.1%}", "",
    f"## 8. Manual Review Rate: {crew_m['manual_review_count']}/{crew_m['total_cases']} = {crew_m['manual_review_rate']:.1%}", "",
    "## 9. Single-agent vs Crew",
    "| Metric | Baseline | Crew |", "|--------|----------|------|",
    f"| Valid output | {base_m['baseline_correct']}/10 | {crew_m['valid_final_count']}/10 |",
    "| Hallucinations caught | 0 | 1 |",
    "| Wrong category corrected | 0 | 1 |",
    "| Ambiguous escalated correctly | 0 | 2 |",
    "| Relative dates flagged | 0 | 1 |", "",
    "## 10. Best Examples",
    "- case_006: HP hallucination caught by Reviewer, cleaned by FallbackHandler",
    "- case_009: wrong category corrected by RepairAgent",
    "- case_004: relative dates flagged, repair marks needs_manual_review", "",
    "## 11. Problematic Examples",
    "- case_003/010: ambiguity unresolvable without LLM -> correct manual_review escalation",
    "- case_007: empty input -> structured safe failure", "",
    "## 12. Next Steps",
    "1. LLM-based disambiguation for ambiguous cases",
    "2. Fuzzy entity matching for typos",
    "3. Confidence threshold check in Reviewer (e.g., flag if conf < 0.4)",
    "4. Differentiate FallbackHandler strategies from Extractor logic",
    "5. Add unit tests per agent",
]

audit_text = "\n".join(audit_lines)
audit_path = ROOT / "docs" / "audit_summary_lab13.md"
audit_path.write_text(audit_text, encoding="utf-8")
print(f"Saved: docs/audit_summary_lab13.md")
print(f"Lines: {len(audit_lines)}")

Saved: docs/audit_summary_lab13.md
Lines: 54
